# Importing Libraries

In [ ]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime, timezone, timedelta
import ast
import pandas as pd

# Setup Configuration

## Load Paths

In [2]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
OUTPUT_PATH = DATA_DIR / "jobs_enriched.csv"

COMPANIES_PATH = DATA_DIR / "companies.csv"

JOBS_RAW_PATH = DATA_DIR / "jobs_raw.csv"



print(f"Base directory: {BASE_DIR} | Base directory exists: {BASE_DIR.exists()}")
print(f"Companies path: {COMPANIES_PATH} |File exists: {COMPANIES_PATH.exists()}")
print(f"Jobs Raw Path: {JOBS_RAW_PATH} | Jobs Raw directory exists: {JOBS_RAW_PATH.parent.exists()}")

Base directory: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai | Base directory exists: True
Companies path: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai\data\companies.csv |File exists: True
Jobs Raw Path: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai\data\jobs_raw.csv | Jobs Raw directory exists: True


## Load Registry


In [3]:
companies_df = pd.read_csv(COMPANIES_PATH).fillna("")
print("Companies loaded:", len(companies_df))
# companies_df.head()

Companies loaded: 6


In [4]:


jobs_df = pd.read_csv(JOBS_RAW_PATH).fillna("")

print("Jobs loaded:", len(jobs_df))


jobs_df.head()

Jobs loaded: 7740


,company,title,location,job_url,description,ats_type,external_job_id,posted_date,date_found,description_raw,updated_date,department,business_unit,work_location_option,posted_datetime,freshness_status
0,OpenAI,"Technical Program Manager, Compute Infrastructure",San Francisco,https://jobs.ashbyhq.com/openai/8fb1615c-34bf-...,About the Team The compute infrastructure team...,ashby,8fb1615c-34bf-47c4-a1d1-b7b2f836bbd3,2026-03-12T16:38:15.322+00:00,2026-06-05T20:48:56+00:00,,,,,,2026-03-12 16:38:15.322000+00:00,old
1,OpenAI,Research Engineer,San Francisco,https://jobs.ashbyhq.com/openai/240d459b-696d-...,"By applying to this role, you will be consider...",ashby,240d459b-696d-43eb-8497-fab3e56ecd9b,2025-04-05T00:03:20.653+00:00,2026-06-05T20:48:56+00:00,,,,,,2025-04-05 00:03:20.653000+00:00,old
2,OpenAI,Account Director - Tokyo,"Tokyo, Japan",https://jobs.ashbyhq.com/openai/18f58952-c242-...,About the team OpenAI’s mission is to build sa...,ashby,18f58952-c242-4562-8732-073a0ae8029e,2026-01-23T00:17:18.483+00:00,2026-06-05T20:48:56+00:00,,,,,,2026-01-23 00:17:18.483000+00:00,old
3,OpenAI,"Software Engineer, RL Training Infra",San Francisco,https://jobs.ashbyhq.com/openai/13995549-e8cc-...,About the Team The Post-Training Frontiers tea...,ashby,13995549-e8cc-498f-9eaa-1869067ac35b,2026-05-23T02:00:50.464+00:00,2026-06-05T20:48:56+00:00,,,,,,2026-05-23 02:00:50.464000+00:00,old
4,OpenAI,"Research Engineer, Retrieval & Search, Applied...",San Francisco,https://jobs.ashbyhq.com/openai/7322d344-9325-...,About the Team We bring OpenAI's technology to...,ashby,7322d344-9325-4a92-8445-0a2c4e9272f8,2024-03-20T21:33:20.763+00:00,2026-06-05T20:48:56+00:00,,,,,,2024-03-20 21:33:20.763000+00:00,old


## Check for missing columns

In [5]:
required_job_columns = [
    "company",
    "title",
    "location",
    "job_url",
    "description",
    "ats_type",
    "external_job_id",
    "posted_date",
    "posted_datetime",
    "freshness_status",
    "date_found"
]

missing_columns = [
    col for col in required_job_columns
    if col not in jobs_df.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required job columns are present.")

All required job columns are present.


## Inspect freshness status

In [6]:
jobs_df["freshness_status"].value_counts()

freshness_status
old        7051
unknown     500
fresh       189
Name: count, dtype: int64

In [7]:
jobs_df.groupby("ats_type").agg(
    total_jobs=("title", "count"),
    fresh_jobs=("freshness_status", lambda x: (x == "fresh").sum()),
    unknown_jobs=("freshness_status", lambda x: (x == "unknown").sum()),
    old_jobs=("freshness_status", lambda x: (x == "old").sum())
)

,total_jobs,fresh_jobs,unknown_jobs,old_jobs
ats_type,,,,
amazon_jobs,150,97,0,53
apple_html,60,0,60,0
ashby,1766,29,0,1737
custom_astrazeneca,2,0,2,0
custom_atlassian,168,0,168,0
custom_bcg,1,0,1,0
custom_bloomberg,36,0,36,0
custom_deloitte,30,0,30,0
custom_microsoft,20,0,20,0


## End


# Helper Functions

## Normalize keywords

In [8]:
def normalize_text(*values):
    """
    Combines multiple values into one lowercase searchable string.
    """
    return " ".join([str(v) for v in values if v is not None]).lower()

## Split Keywords

In [9]:
def split_keywords(keyword_string):
    """
    Converts semicolon-separated keywords into a clean lowercase list.
    Example:
    'machine learning;data scientist;ai engineer'
    ->
    ['machine learning', 'data scientist', 'ai engineer']
    """
    if not keyword_string:
        return []

    return [
        keyword.strip().lower()
        for keyword in str(keyword_string).split(";")
        if keyword.strip()
    ]


## Keyword match

In [10]:
def keyword_in_text(keyword, text):
    """
    Safer keyword match.
    - For multi-word phrases: normal substring match.
    - For single words: word-boundary regex, so 'api' does not match random text badly.
    """
    keyword = str(keyword).lower().strip()
    text = str(text).lower()

    if not keyword:
        return False

    if " " in keyword:
        return keyword in text

    pattern = r"\b" + re.escape(keyword) + r"\b"
    return re.search(pattern, text) is not None

In [11]:
def find_keyword_hits(text, keywords):
    """
    Returns the list of keywords found in text.
    """
    return [
        keyword
        for keyword in keywords
        if keyword_in_text(keyword, text)
    ]

## company-specific keyword matching

In [12]:
# Build company keyword map
company_keywords = {
    row["company"]: split_keywords(row["keywords"])
    for _, row in companies_df.iterrows()
}

company_keywords

{'OpenAI': ['data analyst',
  'machine learning',
  'data scientist',
  'software engineer',
  'data engineer',
  'applied ai',
  'applied ml'],
 'Anthropic': ['data analyst',
  'machine learning',
  'data scientist',
  'software engineer',
  'data engineer',
  'applied ai',
  'applied ml'],
 'Workday': ['data analyst',
  'machine learning',
  'data scientist',
  'software engineer',
  'data engineer',
  'applied ai',
  'applied ml'],
 'Spotify': ['data analyst',
  'machine learning',
  'data scientist',
  'software engineer',
  'data engineer',
  'applied ai',
  'applied ml'],
 'Netflix': ['data analyst',
  'machine learning',
  'data scientist',
  'software engineer',
  'data engineer',
  'applied ai',
  'applied ml'],
 'ThermoFisher': ['data analyst',
  'machine learning',
  'data scientist',
  'software engineer',
  'data engineer',
  'applied ai',
  'applied ml']}

In [13]:
def match_company_keywords(row):
    company = row.get("company", "")
    keywords = company_keywords.get(company, [])

    searchable_text = normalize_text(
        row.get("title", ""),
        row.get("description", ""),
        row.get("location", "")
    )

    return find_keyword_hits(searchable_text, keywords)

## Profile Keywords

In [14]:
target_role_keywords = [
    "machine learning engineer",
    "ml engineer",
    "ai engineer",
    "applied ai",
    "applied scientist",
    "data scientist",
    "data science",
    "ml data engineer",
    "analytics engineer",
    "data engineer",
    "knowledge graph",
    "graph machine learning",
    "research engineer",
    "computer vision engineer",
    "nlp engineer",
    "software engineer",
    "ai data scientist"
]

In [15]:
target_skill_keywords = [
    "python",
    "sql",
    "machine learning",
    "deep learning",
    "pytorch",
    "tensorflow",
    "scikit-learn",
    "sklearn",
    "pandas",
    "numpy",
    "llm",
    "large language model",
    "generative ai",
    "computer vision",
    "nlp",
    "transformers",
    "hugging face",
    "data pipeline",
    "etl",
    "airflow",
    "docker",
    "aws",
    "azure",
    "gcp",
    "postgresql",
    "neo4j",
    "graph",
    "knowledge graph",
    "flask",
    "api",
    "rest",
    "model evaluation",
    "experimentation"
]

In [16]:
project_relevance_keywords = [
    "computer vision",
    "image",
    "classification",
    "detection",
    "llm",
    "generative ai",
    "graph",
    "knowledge graph",
    "fraud",
    "anomaly detection",
    "data pipeline",
    "machine learning",
    "research",
    "nlp"
]

In [17]:
senior_level_keywords = [
    "senior",
    "staff",
    "principal",
    "lead",
    "manager",
    "director",
    "head of",
    "vp",
    "vice president",
    "10+ years",
    "8+ years",
    "7+ years",
    "6+ years",
    "5+ years"
]

In [18]:
good_level_keywords = [
    "new grad",
    "early career",
    "entry level",
    "associate",
    "junior",
    "university graduate",
    "graduate",
    "0+ years",
    "1+ years",
    "2+ years",
]

## Role Score

In [19]:
def calculate_role_score(row):
    title_text = normalize_text(row.get("title", ""))
    full_text = normalize_text(
        row.get("title", ""),
        row.get("description", "")
    )

    title_role_hits = find_keyword_hits(title_text, target_role_keywords)
    all_role_hits = find_keyword_hits(full_text, target_role_keywords)

    score = 0

    if title_role_hits:
        score += 20
    elif all_role_hits:
        score += 12

    score += min(len(all_role_hits) * 2, 5)

    return min(score, 25), all_role_hits

## Skill score

In [20]:
def calculate_skill_score(row):
    full_text = normalize_text(
        row.get("title", ""),
        row.get("description", "")
    )

    skill_hits = find_keyword_hits(full_text, target_skill_keywords)

    score = len(skill_hits) * 3

    return min(score, 35), skill_hits

## Project relevance score

In [21]:
def calculate_project_relevance_score(row):
    full_text = normalize_text(
        row.get("title", ""),
        row.get("description", "")
    )

    project_hits = find_keyword_hits(full_text, project_relevance_keywords)

    score = len(project_hits) * 3

    return min(score, 15), project_hits

## Experience score

In [22]:
def calculate_experience_score(row):
    title_text = normalize_text(row.get("title", ""))
    full_text = normalize_text(
        row.get("title", ""),
        row.get("description", "")
    )

    senior_hits = find_keyword_hits(full_text, senior_level_keywords)
    good_level_hits = find_keyword_hits(full_text, good_level_keywords)

    score = 10

    if good_level_hits:
        score += 5

    # Strong penalty if seniority appears in title
    title_senior_hits = find_keyword_hits(title_text, senior_level_keywords)

    if title_senior_hits:
        score -= 10
    elif senior_hits:
        score -= 6

    score = max(score, 0)

    return min(score, 15), good_level_hits, senior_hits

## Freshness Score

In [23]:
def calculate_freshness_score(row):
    freshness = str(row.get("freshness_status", "")).lower()

    if freshness == "fresh":
        return 10

    if freshness == "unknown":
        return 6

    return 0

## Required check

In [24]:
list_columns = [
    "matched_keywords",
    "matched_roles",
    "matched_skills",
    "project_relevance_hits",
    "missing_keywords",
    "seniority_flags",
    "good_level_hits"
]

for col in list_columns:
    if col in jobs_df.columns:
        jobs_df[col] = jobs_df[col].apply(
            lambda x: "; ".join(x) if isinstance(x, list) else x
        )

## Final Score

In [25]:
def calculate_ats_score(row):
    role_score, matched_roles = calculate_role_score(row)
    skill_score, matched_skills = calculate_skill_score(row)
    project_score, project_hits = calculate_project_relevance_score(row)
    experience_score, good_level_hits, seniority_flags = calculate_experience_score(row)
    freshness_score = calculate_freshness_score(row)

    ats_match_score = (
        role_score
        + skill_score
        + project_score
        + experience_score
        + freshness_score
    )

    missing_keywords = [
        keyword
        for keyword in target_skill_keywords
        if keyword not in matched_skills
    ]

    if ats_match_score >= 80:
        score_label = "Strong Match"
    elif ats_match_score >= 65:
        score_label = "Good Match"
    elif ats_match_score >= 50:
        score_label = "Maybe"
    else:
        score_label = "Low Match"

    score_reasons = []

    if matched_roles:
        score_reasons.append(
            "Role match: " + ", ".join(matched_roles[:5])
        )

    if matched_skills:
        score_reasons.append(
            "Skills matched: " + ", ".join(matched_skills[:8])
        )

    if project_hits:
        score_reasons.append(
            "Project relevance: " + ", ".join(project_hits[:5])
        )

    if seniority_flags:
        score_reasons.append(
            "Seniority concern: " + ", ".join(seniority_flags[:5])
        )

    if not score_reasons:
        score_reasons.append("Limited match based on current scoring rules.")

    return {
        "role_score": role_score,
        "skill_score": skill_score,
        "project_score": project_score,
        "experience_score": experience_score,
        "freshness_score": freshness_score,
        "ats_match_score": ats_match_score,
        "score_label": score_label,
        "matched_roles": matched_roles,
        "matched_skills": matched_skills,
        "project_relevance_hits": project_hits,
        "missing_keywords": missing_keywords[:15],
        "seniority_flags": seniority_flags,
        "good_level_hits": good_level_hits,
        "score_reason": " | ".join(score_reasons)
    }

## Change lists to ;

In [32]:
SIGNAL_COLUMNS = [
    "matched_roles",
    "matched_skills",
    "project_relevance_hits",
    "missing_keywords",
    "seniority_flags",
    "good_level_hits",
    "matched_keywords",
]



In [33]:

def clean_signal_list(value):
    """
    Converts list-like values into clean semicolon text for CSV/PostgreSQL.

    Example:
    ['python', 'sql', 'machine learning']
    becomes:
    python; sql; machine learning
    """

    if value is None:
        return ""

    if isinstance(value, float) and pd.isna(value):
        return ""

    # Actual Python list/set/tuple
    if isinstance(value, (list, tuple, set)):
        cleaned = [
            str(item).strip().lower()
            for item in value
            if str(item).strip()
        ]
        return "; ".join(sorted(set(cleaned)))

    # Already a string
    if isinstance(value, str):
        text = value.strip()

        if text.lower() in ["", "none", "nan", "null", "[]"]:
            return ""

        # If it is accidentally already stored as "['python', 'sql']"
        if text.startswith("[") and text.endswith("]"):
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, (list, tuple, set)):
                    cleaned = [
                        str(item).strip().lower()
                        for item in parsed
                        if str(item).strip()
                    ]
                    return "; ".join(sorted(set(cleaned)))
            except Exception:
                pass

        return text

    return str(value).strip()

## End

# Run 

## Keyword Match Count

In [26]:
jobs_df["matched_keywords"] = jobs_df.apply(
    match_company_keywords,
    axis=1
)

jobs_df["keyword_match_count"] = jobs_df["matched_keywords"].apply(len)

jobs_df[
    [
        "company",
        "title",
        "matched_keywords",
        "keyword_match_count"
    ]
].head(20)

,company,title,matched_keywords,keyword_match_count
0,OpenAI,"Technical Program Manager, Compute Infrastructure",[applied ai],1
1,OpenAI,Research Engineer,[machine learning],1
2,OpenAI,Account Director - Tokyo,[],0
3,OpenAI,"Software Engineer, RL Training Infra",[software engineer],1
4,OpenAI,"Research Engineer, Retrieval & Search, Applied...",[machine learning],1
5,OpenAI,"Researcher, Robustness & Safety Training",[machine learning],1
6,OpenAI,"Software Engineer, Data Infrastructure","[machine learning, software engineer]",2
7,OpenAI,Training: ML Framework Engineer,"[machine learning, software engineer]",2
8,OpenAI,"Software Engineer, Developer Productivity","[software engineer, applied ai]",2
9,OpenAI,"Software Engineer, Data Acquisition",[software engineer],1


## Score all jobs

In [27]:
score_results = jobs_df.apply(calculate_ats_score, axis=1)
score_df = pd.DataFrame(score_results.tolist())

jobs_df = pd.concat(
    [
        jobs_df.reset_index(drop=True),
        score_df.reset_index(drop=True)
    ],
    axis=1
)

jobs_df[
    [
        "company",
        "title",
        "location",
        "freshness_status",
        "ats_match_score",
        "score_label",
        "matched_skills",
        "seniority_flags",
        "job_url"
    ]
].sort_values("ats_match_score", ascending=False).head(25)

,company,title,location,freshness_status,ats_match_score,score_label,matched_skills,seniority_flags,job_url
7311,Snowflake,Applied AI Engineer,PL-Warsaw,old,80,Strong Match,"[python, sql, machine learning, pandas, numpy,...",[],https://jobs.ashbyhq.com/snowflake/467250b0-43...
7567,Twilio,Machine Learning Engineer,Remote - US,old,79,Good Match,"[python, sql, machine learning, deep learning,...","[lead, 5+ years]",https://job-boards.greenhouse.io/twilio/jobs/7...
6014,Scale AI,Senior Machine Learning Engineer - Model Evalu...,"San Francisco, CA; St. Louis, MO; New York, NY...",old,77,Good Match,"[python, machine learning, deep learning, pyto...","[senior, director]",https://job-boards.greenhouse.io/scaleai/jobs/...
2169,Airbnb,"Senior Data Scientist, Guest Travel Insurance ...",United States,old,75,Good Match,"[python, sql, machine learning, deep learning,...","[senior, lead, manager, 5+ years]",https://careers.airbnb.com/positions/7926614?g...
4491,Reddit,"Staff Machine Learning Engineer, Consumer",Remote - United States,old,75,Good Match,"[python, machine learning, deep learning, pyto...","[senior, staff, lead, 7+ years]",https://job-boards.greenhouse.io/reddit/jobs/7...
6015,Scale AI,"Senior Machine Learning Engineer, Public Sector","San Francisco, CA; New York, NY; Washington, DC",old,75,Good Match,"[python, machine learning, deep learning, pyto...","[senior, director]",https://job-boards.greenhouse.io/scaleai/jobs/...
1183,Spotify,"Machine Learning Engineer, Personalization, Mi...","New York, NY US remote",old,74,Good Match,"[python, sql, machine learning, pytorch, tenso...",[],https://jobs.lever.co/spotify/de3f6a47-4d75-45...
1182,Spotify,"Machine Learning Engineer I, Personalization ,...","New York, NY US remote",old,74,Good Match,"[python, sql, machine learning, pytorch, tenso...",[],https://jobs.lever.co/spotify/fd79c3f5-1b2c-47...
2232,Airbnb,"Senior Staff Machine Learning Engineer, Growth...",Remote - USA,old,72,Good Match,"[python, machine learning, deep learning, pyto...","[senior, staff, manager]",https://careers.airbnb.com/positions/7747259?g...
4443,Reddit,"Senior Machine Learning Engineer, GenAI Security",Remote - United States,old,72,Good Match,"[python, machine learning, deep learning, pyto...","[senior, lead, 5+ years]",https://job-boards.greenhouse.io/reddit/jobs/7...


## Relevancy tag

In [28]:
jobs_df["is_relevant"] = (
    jobs_df["freshness_status"].isin(["fresh", "unknown"])
    & (jobs_df["keyword_match_count"] > 0)
    & (jobs_df["ats_match_score"] >= 50)
)

print("Total jobs:", len(jobs_df))
print("Relevant jobs:", jobs_df["is_relevant"].sum())
print("Strong matches:", (jobs_df["ats_match_score"] >= 80).sum())

Total jobs: 7740
Relevant jobs: 4
Strong matches: 1


## list to ;

In [34]:
for col in SIGNAL_COLUMNS:
    if col in jobs_df.columns:
        jobs_df[col] = jobs_df[col].apply(clean_signal_list)

## View

In [35]:
strong_matches_df = jobs_df[
    jobs_df["ats_match_score"] >= 60
].sort_values("ats_match_score", ascending=False)

strong_matches_df[
    [
        "company",
        "title",
        "location",
        "freshness_status",
        "ats_match_score",
        "score_label",
        "score_reason",
        "job_url"
    ]
].head(30)

,company,title,location,freshness_status,ats_match_score,score_label,score_reason,job_url
7311,Snowflake,Applied AI Engineer,PL-Warsaw,old,80,Strong Match,"Role match: ai engineer, applied ai, data scie...",https://jobs.ashbyhq.com/snowflake/467250b0-43...
7567,Twilio,Machine Learning Engineer,Remote - US,old,79,Good Match,"Role match: machine learning engineer, data sc...",https://job-boards.greenhouse.io/twilio/jobs/7...
6014,Scale AI,Senior Machine Learning Engineer - Model Evalu...,"San Francisco, CA; St. Louis, MO; New York, NY...",old,77,Good Match,"Role match: machine learning engineer, ml engi...",https://job-boards.greenhouse.io/scaleai/jobs/...
4491,Reddit,"Staff Machine Learning Engineer, Consumer",Remote - United States,old,75,Good Match,"Role match: machine learning engineer, ml engi...",https://job-boards.greenhouse.io/reddit/jobs/7...
2169,Airbnb,"Senior Data Scientist, Guest Travel Insurance ...",United States,old,75,Good Match,Role match: data scientist | Skills matched: p...,https://careers.airbnb.com/positions/7926614?g...
6015,Scale AI,"Senior Machine Learning Engineer, Public Sector","San Francisco, CA; New York, NY; Washington, DC",old,75,Good Match,Role match: machine learning engineer | Skills...,https://job-boards.greenhouse.io/scaleai/jobs/...
1183,Spotify,"Machine Learning Engineer, Personalization, Mi...","New York, NY US remote",old,74,Good Match,"Role match: machine learning engineer, ml engi...",https://jobs.lever.co/spotify/de3f6a47-4d75-45...
1182,Spotify,"Machine Learning Engineer I, Personalization ,...","New York, NY US remote",old,74,Good Match,"Role match: machine learning engineer, ml engi...",https://jobs.lever.co/spotify/fd79c3f5-1b2c-47...
2232,Airbnb,"Senior Staff Machine Learning Engineer, Growth...",Remote - USA,old,72,Good Match,"Role match: machine learning engineer, ai engi...",https://careers.airbnb.com/positions/7747259?g...
4443,Reddit,"Senior Machine Learning Engineer, GenAI Security",Remote - United States,old,72,Good Match,Role match: machine learning engineer | Skills...,https://job-boards.greenhouse.io/reddit/jobs/7...


In [36]:
jobs_df[
    [
        "company",
        "title",
        "location",
        "freshness_status",
        "ats_match_score",
        "score_label",
        "matched_skills",
        "seniority_flags",
        "job_url"
    ]
].sort_values("ats_match_score", ascending=False).head(25)

,company,title,location,freshness_status,ats_match_score,score_label,matched_skills,seniority_flags,job_url
7311,Snowflake,Applied AI Engineer,PL-Warsaw,old,80,Strong Match,aws; azure; etl; gcp; generative ai; llm; mach...,,https://jobs.ashbyhq.com/snowflake/467250b0-43...
7567,Twilio,Machine Learning Engineer,Remote - US,old,79,Good Match,airflow; aws; azure; data pipeline; deep learn...,5+ years; lead,https://job-boards.greenhouse.io/twilio/jobs/7...
6014,Scale AI,Senior Machine Learning Engineer - Model Evalu...,"San Francisco, CA; St. Louis, MO; New York, NY...",old,77,Good Match,aws; computer vision; deep learning; gcp; llm;...,director; senior,https://job-boards.greenhouse.io/scaleai/jobs/...
2169,Airbnb,"Senior Data Scientist, Guest Travel Insurance ...",United States,old,75,Good Match,airflow; computer vision; data pipeline; deep ...,5+ years; lead; manager; senior,https://careers.airbnb.com/positions/7926614?g...
4491,Reddit,"Staff Machine Learning Engineer, Consumer",Remote - United States,old,75,Good Match,airflow; data pipeline; deep learning; experim...,7+ years; lead; senior; staff,https://job-boards.greenhouse.io/reddit/jobs/7...
6015,Scale AI,"Senior Machine Learning Engineer, Public Sector","San Francisco, CA; New York, NY; Washington, DC",old,75,Good Match,aws; computer vision; deep learning; gcp; gene...,director; senior,https://job-boards.greenhouse.io/scaleai/jobs/...
1183,Spotify,"Machine Learning Engineer, Personalization, Mi...","New York, NY US remote",old,74,Good Match,aws; data pipeline; experimentation; gcp; larg...,,https://jobs.lever.co/spotify/de3f6a47-4d75-45...
1182,Spotify,"Machine Learning Engineer I, Personalization ,...","New York, NY US remote",old,74,Good Match,aws; data pipeline; experimentation; gcp; larg...,,https://jobs.lever.co/spotify/fd79c3f5-1b2c-47...
2232,Airbnb,"Senior Staff Machine Learning Engineer, Growth...",Remote - USA,old,72,Good Match,airflow; computer vision; data pipeline; deep ...,manager; senior; staff,https://careers.airbnb.com/positions/7747259?g...
4443,Reddit,"Senior Machine Learning Engineer, GenAI Security",Remote - United States,old,72,Good Match,airflow; data pipeline; deep learning; etl; ex...,5+ years; lead; senior,https://job-boards.greenhouse.io/reddit/jobs/7...


## End


# Save CSV

In [38]:


jobs_df.to_csv(OUTPUT_PATH, index=False)

print("Saved enriched jobs to:", OUTPUT_PATH)
print("Total jobs:", len(jobs_df))
print("Relevant jobs:", jobs_df["is_relevant"].sum())
print("Strong matches:", (jobs_df["ats_match_score"] >= 80).sum())

Saved enriched jobs to: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai\data\jobs_enriched.csv
Total jobs: 7740
Relevant jobs: 4
Strong matches: 1


In [31]:
jobs_df[
    [
        "company",
        "title",
        "ats_type",
        "freshness_status",
        "keyword_match_count",
        "ats_match_score",
        "score_label",
        "is_relevant"
    ]
].sort_values("ats_match_score", ascending=False).head(20)

,company,title,ats_type,freshness_status,keyword_match_count,ats_match_score,score_label,is_relevant
7311,Snowflake,Applied AI Engineer,ashby,old,0,80,Strong Match,False
7567,Twilio,Machine Learning Engineer,greenhouse,old,0,79,Good Match,False
6014,Scale AI,Senior Machine Learning Engineer - Model Evalu...,greenhouse,old,0,77,Good Match,False
2169,Airbnb,"Senior Data Scientist, Guest Travel Insurance ...",greenhouse,old,0,75,Good Match,False
4491,Reddit,"Staff Machine Learning Engineer, Consumer",greenhouse,old,0,75,Good Match,False
6015,Scale AI,"Senior Machine Learning Engineer, Public Sector",greenhouse,old,0,75,Good Match,False
1183,Spotify,"Machine Learning Engineer, Personalization, Mi...",lever,old,2,74,Good Match,False
1182,Spotify,"Machine Learning Engineer I, Personalization ,...",lever,old,2,74,Good Match,False
2232,Airbnb,"Senior Staff Machine Learning Engineer, Growth...",greenhouse,old,0,72,Good Match,False
4443,Reddit,"Senior Machine Learning Engineer, GenAI Security",greenhouse,old,0,72,Good Match,False
